In [2]:
!pip install httpx


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


In [13]:
# If httpx is not installed, run this once in another cell:
# !pip install httpx

import asyncio
import json
import httpx

# =========================
# FILL THESE IN
# =========================
URL = "https://ldc1in003d.tcsgroup.ch:7083/insurance-policies/v1/technical/idit/authorization/token"
JSON_BODY = {
        "invalidToken": "eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICI3WmlhSDlhdW9hc3ppMWI3UHZhWWxmOFMxXzdKdE5wOWJOamtFQTA1SFkwIn0.eyJleHAiOjE3OTE4Mjg5ODEsImlhdCI6MTc5MTgyODY4MSwianRpIjoidHJydGNjOjQ4ZTgyZjY3LTI4YTYtMWZiNy01MmM4LTRmZGVkMTYyYTkxNiIsImlzcyI6Imh0dHBzOi8vdGNzLXRzdC1ha3Mta2V5Y2xvYWsudGNzLnNhcGllbnNtZ3MuY29tL3JlYWxtcy90Y3MtaWRpdC11YXQiLCJhdWQiOlsiYmFja2VuZC10by1iYWNrZW5kIiwiYWNjb3VudCJdLCJzdWIiOiJlYmE1ZmNmYy1mZWZiLTRlYmYtODQxNC1iODgyMzk5Y2I1NGIiLCJ0eXAiOiJCZWFyZXIiLCJhenAiOiJiYWNrZW5kLXRvLWJhY2tlbmQiLCJhY3IiOiIxIiwiYWxsb3dlZC1vcmlnaW5zIjpbIioiXSwicmVhbG1fYWNjZXNzIjp7InJvbGVzIjpbImRlZmF1bHQtcm9sZXMtdGNzLWlkaXQtdWF0Iiwib2ZmbGluZV9hY2Nlc3MiLCJ1bWFfYXV0aG9yaXphdGlvbiJdfSwicmVzb3VyY2VfYWNjZXNzIjp7ImlkaXQtYXBpIjp7InJvbGVzIjpbInVtYV9wcm90ZWN0aW9uIl19LCJhY2NvdW50Ijp7InJvbGVzIjpbIm1hbmFnZS1hY2NvdW50IiwibWFuYWdlLWFjY291bnQtbGlua3MiLCJ2aWV3LXByb2ZpbGUiXX19LCJzY29wZSI6InByb2ZpbGUgZW1haWwiLCJjbGllbnRIb3N0IjoiMTAuODcuMi4yMzgiLCJlbWFpbF92ZXJpZmllZCI6ZmFsc2UsImF6cF9pZGl0IjoiYmFja2VuZC10by1iYWNrZW5kIiwicHJlZmVycmVkX3VzZXJuYW1lIjoic2VydmljZS1hY2NvdW50LWlkaXQtYXBpIiwiY2xpZW50QWRkcmVzcyI6IjEwLjg3LjIuMjM4IiwiY2xpZW50X2lkIjoiaWRpdC1hcGkifQ.gzBkWJ1a-_yd-qmeWk59jCJ0rg04xwNlto0L60xjteFcZgC5SF2_TC0IN9hbjsG-CPdFWAPgaQSD8zLhQHCQMLJJxg_p3hNKtv49M0IXF-hKVlBqrEnA1D5H9l5kjZH6BDN46OxgdSOrnaRY02yWikYnUA5m3laDtPhn6-9N02HwqrrbtxmLNM09AxOkWzOpH6fhUu5GoZlUrsx66uLtYSRRCgr522qx5-byV_StuG3_hYwlrhAPQRUZR3MP31VGbVuOeywAWeBbSIgXDTAniGrDe4Da8xjib39Zd0aAGzixwtOH8zGrcibY5TRoClYwI7NHdeg-vKhiW7Qj3DM6lA"
}
HEADERS = {
    "Content-Type": "application/json"
    # add auth headers here if needed
}
OUTPUT_FILE = "results.txt"
NUM_CALLS = 50
# =========================

file_lock = asyncio.Lock()

async def call_api(call_number, client):
    try:
        response = await client.post(URL, json=JSON_BODY, headers=HEADERS)

        try:
            result = json.dumps(response.json(), indent=2, ensure_ascii=False)
        except Exception:
            result = response.text

        entry = (
            f"CALL {call_number}\n"
            f"STATUS: {response.status_code}\n"
            f"{result}"
        )

    except Exception as e:
        entry = (
            f"CALL {call_number}\n"
            f"ERROR: {str(e)}"
        )

    async with file_lock:
        with open(OUTPUT_FILE, "a", encoding="utf-8") as f:
            f.write(entry)
            f.write("\n\n")

async def run_calls():
    async with httpx.AsyncClient(timeout=60.0, verify=False) as client:
        tasks = [call_api(i, client) for i in range(1, NUM_CALLS + 1)]
        await asyncio.gather(*tasks)

await run_calls()
print(f"Done. Appended {NUM_CALLS} call results to {OUTPUT_FILE}")

Done. Appended 50 call results to results.txt
